# Requirements

In [1]:
!pip install ag2[ollama]
!pip install autogen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.9/855.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.7 MB/s eta 0:00:00


# Setup Environment
Install system dependencies and start Ollama in the background.

In [2]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,374 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,287 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,812 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:14 http:

In [3]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [4]:
import os
import json
import time
import subprocess

# Pull LLM Model
Pull the Llama 3.2 model for local inference.

In [5]:
!ollama pull llama3.2

# LLM Configuration
Configure AutoGen to use Ollama.

In [6]:
llm_config = {
    "model": "llama3.2",
    "api_type": "ollama",
    "base_url": "http://localhost:11434",
    "api_key": "ollama"  # Dummy key, as Ollama doesn't require one
}

# Mount Google Drive
Mount Drive for persistent storage of research outputs.

In [7]:
from google.colab import drive
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

try:
    drive.mount('/content/drive')
    logging.info("Google Drive mounted successfully.")
except Exception as e:
    logging.error(f"Failed to mount Drive: {e}")
    raise

Mounted at /content/drive


# Research Storage
Initialize and manage a JSON file for storing research summaries and answers.

In [8]:
# Initialize JSON file for research storage
research_file = "/content/drive/MyDrive/research_outputs.json"
if not os.path.exists(research_file):
    with open(research_file, "w") as f:
        json.dump({"summaries": [], "answers": []}, f)

def save_research_output(output_type, content):
    try:
        with open(research_file, "r+") as f:
            data = json.load(f)
            data[output_type].append({"content": content, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")})
            f.seek(0)
            json.dump(data, f, indent=4)
        print(f"{output_type.capitalize()} saved: {content[:50]}...")
    except Exception as e:
        print(f"Error saving {output_type}: {e}")

# Define agents
Define the multi-agent system using AutoGen. Each agent has a specific role:

* **QueryAgent**: Interprets and assigns tasks.
* **SummaryAgent**: Summarizes texts and saves outputs.
* **AnswerAgent**: Provides detailed answers and saves them.
* **StorageAgent**: Verifies and ensures storage.
* **UserProxy**: Represents the user for input.

In [10]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
query_agent = AssistantAgent(
    name="QueryAgent",
    llm_config=llm_config,
    system_message="You are a research coordinator. Interpret the user's research question or task (e.g., summarize a paper, answer a question). "
    "Assign tasks to the Summary Agent for summarization or the Answer Agent for detailed responses. Example: 'Summarize this abstract: [text]' or "
    "'Answer: How do LLMs enhance RL?'"
)

summary_agent = AssistantAgent(
    name="SummaryAgent",
    llm_config=llm_config,
    system_message="Summarize provided text (e.g., paper abstracts, snippets) into 100–200 words, focusing on key findings, methods, and contributions."
    " Store the summary using save_research_output('summaries', summary). Example: 'Summarize: [text]'"
)

answer_agent = AssistantAgent(
    name="AnswerAgent",
    llm_config=llm_config,
    system_message="Provide detailed, accurate answers to research questions based on general knowledge or provided context. "
    "Store the answer using save_research_output('answers', answer). Example: 'Answer: What is the role of LLMs in autonomous driving?'"
)

storage_agent = AssistantAgent(
    name="StorageAgent",
    llm_config=llm_config,
    system_message="Ensure summaries and answers are stored correctly in the JSON file. Verify storage by checking the file and report any issues."
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="ALWAYS",
    max_consecutive_auto_reply=0
)

# Set Up Group Chat
Create a group chat manager for agent collaboration.

In [14]:
group_chat = GroupChat(
    agents=[query_agent, summary_agent, answer_agent, storage_agent, user_proxy],
    messages=[],
    max_round=5,
    speaker_selection_method= "round_robin"
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config
)

# Start the conversation
Initiate the chat with a sample message. In a real run, this would prompt for user input.

In [15]:
print("Starting Research Agent...")
try:
    user_proxy.initiate_chat(
        manager,
        message="What's the difference between Agnetic AI and AI agents?"
    )
except Exception as e:
    print(f"Error during chat: {e}")

Starting Research Agent...
User (to chat_manager):

What's the difference between Agnetic AI and AI agents?

--------------------------------------------------------------------------------

Next speaker: QueryAgent

QueryAgent (to chat_manager):

AI agents and Agent-Based Models (ABMs) are related concepts, but they serve different purposes.

**Artificial Intelligence (AI) Agents:**
An AI agent is a software component that perceives its environment, takes actions, and learns to improve its performance over time. AI agents can be found in various applications, such as robotics, game playing, or autonomous vehicles. They are designed to interact with their surroundings, make decisions, and adapt to new situations. AI agents typically rely on machine learning algorithms to learn from data and improve their performance.

**Agent-Based Models (ABMs):**
An Agent-Based Model (ABM) is a computational model that simulates the behavior of individuals or entities (agents) interacting with each o